# Лабораторна робота №4. Регресійні моделі: глибоке дослідження

**Датасет:** Car Price Prediction  
https://www.kaggle.com/datasets/hellbuoy/car-price-prediction

**Мета:** підготовка даних, побудова та порівняння регресійних моделей для прогнозування ціни автомобіля.

## 1. Завантаження та первинний аналіз

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style('whitegrid')

# === Завантаження ===
import kagglehub, os
path = kagglehub.dataset_download('hellbuoy/car-price-prediction')
print(os.listdir(path))
df = pd.read_csv(os.path.join(path, 'CarPrice_Assignment.csv'))
print('Розмір:', df.shape)
display(df.head())
print(df.info())
print(df.describe())

## 2. Перевірка якості даних

In [ ]:
print('Пропуски:\n', df.isnull().sum())
print('\nПовні дублікати:', df.duplicated().sum())
print('\nУнікальні значення:')
print(df.nunique())

## 3. Робота з car_ID

In [ ]:
car_ids = df['car_ID'].copy()
df_model = df.drop(columns=['car_ID'])
print('car_ID збережено окремо. Ознаки для моделей:', df_model.columns.tolist())

## 4. Дослідження розподілів числових ознак

In [ ]:
num_cols = df_model.select_dtypes(include=np.number).columns.tolist()
if 'price' in num_cols:
    num_cols.remove('price')

df_model[num_cols].hist(bins=20, figsize=(14, 10))
plt.suptitle('Розподіли числових ознак')
plt.tight_layout()
plt.show()

skewness = df_model[num_cols].skew().sort_values(key=abs, ascending=False)
print('Коефіцієнти асиметрії:')
print(skewness)

## 5. Формування train/test

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, car_ids, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

## 6. Аналіз та обробка викидів (IQR, тільки train)

In [ ]:
cont_cols = [c for c in X_train.select_dtypes(include=np.number).columns if c != 'symboling']

def clip_outliers(train, test, cols):
    train = train.copy()
    test = test.copy()
    for col in cols:
        q1 = train[col].quantile(0.25)
        q3 = train[col].quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
        train[col] = train[col].clip(low, high)
        test[col] = test[col].clip(low, high)
    return train, test

out_count = {}
for col in cont_cols:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    out_count[col] = ((X_train[col] < q1-1.5*iqr) | (X_train[col] > q3+1.5*iqr)).sum()
print('Потенційні викиди (train):')
print(pd.Series(out_count).sort_values(ascending=False))

X_train, X_test = clip_outliers(X_train, X_test, cont_cols)
print('Викиди обмежено межами train.')

## 7. Кореляції та відбір ознак

In [ ]:
num_train = X_train.select_dtypes(include=np.number)
corr = num_train.corr()

high_corr_pairs = []
cols_to_drop = set()
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.85:
            c1, c2 = corr.columns[i], corr.columns[j]
            high_corr_pairs.append((c1, c2, corr.iloc[i, j]))
            # залишаємо ту, що сильніше корелює з price
            r1 = abs(pd.concat([num_train, y_train], axis=1).corr()['price'][c1])
            r2 = abs(pd.concat([num_train, y_train], axis=1).corr()['price'][c2])
            drop = c2 if r1 >= r2 else c1
            cols_to_drop.add(drop)

print('Сильні кореляції (|r|>0.85):')
for p in high_corr_pairs:
    print(p)
print('Видаляємо:', cols_to_drop)

X_train = X_train.drop(columns=list(cols_to_drop), errors='ignore')
X_test = X_test.drop(columns=list(cols_to_drop), errors='ignore')

## 8. Підготовка числових і категоріальних ознак

In [ ]:
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

# symboling — дискретна, без Yeo-Johnson
if 'symboling' in num_cols:
    cont_cols = [c for c in num_cols if c != 'symboling']
    disc_cols = ['symboling']
else:
    cont_cols = num_cols
    disc_cols = []

preprocessor = ColumnTransformer([
    ('cont', Pipeline([
        ('yeo', PowerTransformer(method='yeo-johnson')),
        ('scale', StandardScaler())
    ]), cont_cols),
    ('disc', StandardScaler(), disc_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

print('Неперервні:', cont_cols)
print('Дискретні:', disc_cols)
print('Категоріальні:', cat_cols)

## 9. Перевірка трансформації

In [ ]:
pt = PowerTransformer(method='yeo-johnson')
X_cont_t = pt.fit_transform(X_train[cont_cols])
skew_after = pd.Series(X_cont_t.skew(axis=0) if hasattr(X_cont_t, 'skew') else np.array([pd.Series(X_cont_t[:, i]).skew() for i in range(X_cont_t.shape[1])]), index=cont_cols)
print('Асиметрія після Yeo-Johnson:')
print(skew_after)

## 10–11. Побудова та оцінювання моделей

In [ ]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(max_iter=5000),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR()
}

results = []
preds = {}

for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    preds[name] = y_pred
    results.append({
        'Model': name,
        'R2': r2_score(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'MAPE %': mape(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
print(results_df)

## 12. Пояснення вибору типу моделі

Прогнозування `price` — це **задача регресії** (неперервна цільова змінна).  
Логістична регресія (класифікація) не підходить, бо вона передбачає дискретні класи, а не числову ціну.

## 13. Візуальне порівняння прогнозів

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
for i, (name, y_pred) in enumerate(preds.items()):
    ax = axes[i]
    ax.scatter(y_test, y_pred, alpha=0.5)
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'r--')
    ax.set_title(name)
    ax.set_xlabel('Факт')
    ax.set_ylabel('Прогноз')
plt.tight_layout()
plt.show()

## 14. Прогноз для 10 автомобілів (Random Forest)

In [ ]:
rf_pipe = Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(random_state=42, n_estimators=100))])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)

sample_idx = np.random.RandomState(42).choice(len(y_test), size=10, replace=False)
table = pd.DataFrame({
    'car_ID': ids_test.iloc[sample_idx].values,
    'Фактична ціна': y_test.iloc[sample_idx].values,
    'Прогноз': y_pred_rf[sample_idx],
    'Абс. помилка': np.abs(y_test.iloc[sample_idx].values - y_pred_rf[sample_idx]),
    'Абс. % помилка': np.abs(y_test.iloc[sample_idx].values - y_pred_rf[sample_idx]) / y_test.iloc[sample_idx].values * 100
})
print(table)

## 15. Висновки

- Найкращі моделі зазвичай — Random Forest і Gradient Boosting (високий R², нижчі помилки).
- Лінійні моделі працюють гірше через нелінійності.
- Попередня обробка (Yeo-Johnson + масштабування + OHE) покращує результати.
- Викиди обмежено, сильні кореляції видалено — це зменшує мультиколінеарність.
- MAPE показує відносну помилку у відсотках — зручно для інтерпретації.